In [724]:
import numpy as np
import scipy.optimize as opt
import serial
import serial.tools.list_ports as ports
import time
import can
import struct

# ----- RS485 COMMAND SET -----
def moveTo_XY(x, y):
    """
        Commands X-Y stage to move to specified point in x-y plane
        Inputs:
            x: float: X location to move to in mm (0-550)
            y: float: Y location to move to in mm (0-550)
    """
    if x == readPosition(3) and y == readPosition(4):
        return
         
    #Calculating steps based on distance and physical properties of rails
    steps_X = int(np.abs(x*stepsPerRotation_XY/mmPerThread))
    steps_Y = int(np.abs(y*stepsPerRotation_XY/mmPerThread))

    #match move times   
    if (np.abs(x-readPosition(3))*(np.abs(y-readPosition(4))) != 0):
        speed_X = int(np.min(np.array([maxVel_X, maxVel_Y*(np.abs(x-readPosition(3))/np.abs(y-readPosition(4)))])))
        speed_Y = int(np.min(np.array([maxVel_Y, maxVel_X*(np.abs(y-readPosition(4))/np.abs(x-readPosition(3)))])))
        acc_X = int(np.min(np.array([maxAcc_X, maxAcc_Y*speed_X/speed_Y])))
        acc_Y = int(np.min(np.array([maxAcc_Y, maxAcc_X*speed_Y/speed_X])))
        setSpeedAcc(3, speed_X, acc_X)
        setSpeedAcc(4, speed_Y, acc_Y)
    elif (np.abs(x-readPosition(3)) == 0):
        setSpeedAcc(4, maxVel_Y, maxAcc_Y)
    elif (np.abs(y-readPosition(4)) == 0):
        setSpeedAcc(3, maxVel_X, maxAcc_X)
    
    moveString_X = f"/3A{str(steps_X)}<CR>\r\n"
    moveString_Y = f"/4A{str(steps_Y)}<CR>\r\n"
    runString = f"/CR\r\n"
    
    #Command Motors
    ser.write(moveString_X.encode('ascii'))
    time.sleep(0.1)
    ser.write(moveString_Y.encode('ascii'))
    time.sleep(0.1)
    ser.write(runString.encode('ascii'))
    time.sleep(0.1)
    print(f"XY command sent. Moving to ({x}, {y})")


def moveRelative_XY(x, y):
    """
        Commands X-Y stage to move to specified point in x-y plane relative to its current position
        Inputs:
            x: float: X distance to move to in mm 
            y: float: Y distance to move to in mm 
    """
    if x == 0 and y == 0:
        return
        
    #Calculating steps based on distance and physical properties of rails
    steps_X = int(abs(x*stepsPerRotation_XY/mmPerThread))
    steps_Y = int(abs(y*stepsPerRotation_XY/mmPerThread))

    #match move times
    if x*y !=0:
        speed_X = int(np.min(np.array([maxVel_X, maxVel_Y*(np.abs(x)/np.abs(y))])))
        speed_Y = int(np.min(np.array([maxVel_Y, maxVel_X*(np.abs(y)/np.abs(x))])))
        acc_X = int(np.min(np.array([maxAcc_X, maxAcc_Y*speed_X/speed_Y])))
        acc_Y = int(np.min(np.array([maxAcc_Y, maxAcc_X*speed_Y/speed_X])))
        setSpeedAcc(3, speed_X, acc_X)
        setSpeedAcc(4, speed_Y, acc_Y)
    elif (x == 0):
        setSpeedAcc(4, maxVel_Y, maxAcc_Y)
    elif (y == 0):
        setSpeedAcc(3, maxVel_X, maxAcc_X)
        
    #Create String to send
    if x>0:
        moveString_X = f"/3P{str(steps_X)}<CR>\r\n"
    elif x<0:
        moveString_X = f"/3D{str(steps_X)}<CR>\r\n"
    else:
        moveString_X = f"/3P1<CR>\r\n" #Find a way to fix this ideally. when the realtive position is set to zero weird things happen, so i just make it 1 artifiacially so it barely moves
    if y>0:
        moveString_Y = f"/4P{str(steps_Y)}<CR>\r\n"
    elif y<0:
        moveString_Y = f"/4D{str(steps_Y)}<CR>\r\n"
    else:
        moveString_Y = f"/4P1<CR>\r\n"
    runString = f"/CR\r\n"

    #Command Motors
    ser.write(moveString_X.encode('ascii'))
    time.sleep(0.1)
    ser.write(moveString_Y.encode('ascii'))
    time.sleep(0.1)
    print(f"XY command sent. Moving by ({x}, {y}) to ({x + readPosition(3)}, {y + readPosition(4)})")
    ser.write(runString.encode('ascii'))
    time.sleep(0.1)

def moveTo_Ze(theta):
    """
        Commands zenith motor to move to specified point 
        Inputs:
            theta: float: location to move to (degrees)
    """
    steps_Ze = int(200*256*theta/360) #steps*microsteps*angle in degrees
    
    #Create String to send
    MoveString_Ze = f"/1A{str(steps_Ze)}<CR>\r\n"
    runString = f"/1R\r\n"

    #Command Motors
    ser.write(MoveString_Ze.encode('ascii'))
    time.sleep(0.05)
    ser.write(runString.encode('ascii'))
    time.sleep(0.05)
    print(f"Ze command sent. Moving zenith axis to {round(theta, 3)} degrees")


def moveRelative_Ze(theta):
    """
        Commands Zenith motor to move a certain amount from its current position
        Inputs:
            theta: float: distance to move in degrees
    """
    if theta == 0:
        return
    
    steps_Ze = int(200*256*theta/360) # steps*microsteps*ratio of how much of a circle we are going in
    setSpeedAcc(1, int(maxVel_Ze), maxAcc_Ze)
    
    #Create String to send
    moveString_Ze = f"/1P{steps_Ze}<CR>\r\n"
    runString = f"/1R\r\n"

    #Command Motors
    ser.write(moveString_Ze.encode('ascii'))
    time.sleep(0.1)
    ser.write(runString.encode('ascii'))
    time.sleep(0.5)
    print(f"Ze command sent. Moving zenith axis by {round(theta, 3)} degrees")   


def moveTo_all(x, y):
    """
        Moves all motors so that the the heat source is pointed towards a target some distance (H defined below) above (x, y) = (275, 275)
        Inputs:
            x: float: X location to move to in mm (0-550)
            y: float: Y location to move to in mm (0-550)
    """
    #Calculating steps based on distance and physical properties of rails---------------------------------------------
    steps_X = int(np.abs(x*stepsPerRotation_XY/mmPerThread))
    steps_Y = int(np.abs(y*stepsPerRotation_XY/mmPerThread))

    # matching Move times---------------------------------------------------------------------------------------------
    if (np.abs(x-readPosition(3)) >= np.abs(y-readPosition(4))) and (np.abs(x-readPosition(3))!=0):
        setSpeedAcc(4, int(np.min(np.array([maxVel_Y, maxVel_X*(np.abs(y-readPosition(4))/np.abs(x-readPosition(3)))]))), maxAcc_Y)
        setSpeedAcc(3, int(np.min(np.array([maxVel_X, maxVel_Y*(np.abs(x-readPosition(3))/np.abs(y-readPosition(4)))]))), maxAcc_X)
        travelTime = abs(steps_X-int(np.abs(readPosition(3)*stepsPerRotation_XY/mmPerThread)))/maxVel_X
    elif (np.abs(x-readPosition(3)) <= np.abs(y-readPosition(4))) and (np.abs(y-readPosition(4))!=0):
        setSpeedAcc(3, int(np.min(np.array([maxVel_X, maxVel_Y*(np.abs(x-readPosition(3))/np.abs(y-readPosition(4)))]))), maxAcc_X)
        setSpeedAcc(4, int(np.min(np.array([maxVel_Y, maxVel_X*(np.abs(y-readPosition(4))/np.abs(x-readPosition(3)))]))), maxAcc_Y)
        travelTime = abs(steps_Y-int(np.abs(readPosition(4)*stepsPerRotation_XY/mmPerThread)))/maxVel_Y
    elif (np.abs(x-readPosition(3)) == 0):
        setSpeedAcc(4, maxVel_Y, maxAcc_Y)
        travelTime = abs(steps_Y-int(np.abs(readPosition(4)*stepsPerRotation_XY/mmPerThread)))/maxVel_Y
    elif (np.abs(y-readPosition(4)) == 0):
        setSpeedAcc(3, maxVel_X, maxAcc_X)
        travelTime = abs(steps_X-int(np.abs(readPosition(3)*stepsPerRotation_XY/mmPerThread)))/maxVel_X
    if (np.abs(x-readPosition(3)) == 0) and (np.abs(y-readPosition(4)) == 0):
        travelTime = 1

    if readPosition(4) != 275:
        old_Az = int(-np.round(np.arctan2((readPosition(3)-275), (readPosition(4)-275))*6*3200/(2*np.pi)-(2/8)*6*3200))
    else:
        old_Az = int(-np.round((np.pi/2)*6*3200/(2*np.pi)-(2/8)*6*3200))
    old_Ze = findTheta(readPosition(3), readPosition(4))*180/np.pi
    if y != 275:
        new_Az = int(-np.round(np.arctan2((x-275), (y-275))*6*3200/(2*np.pi)-(2/8)*6*3200))
    else:
        new_Az = int(-np.round((np.pi/2)*6*3200/(2*np.pi)-(2/8)*6*3200))
    new_Ze = findTheta(x, y)*180/np.pi

    if readPosition(3) == x and readPosition(4) == y:
        setSpeedAcc(2, maxVel_Az, maxAcc_Az)
        time.sleep(0.1)

        setSpeedAcc(1, maxVel_Ze, maxAcc_Ze)
        time.sleep(0.1)
    else:
        setSpeedAcc(2, (int(np.abs(old_Az - new_Az)/travelTime)), maxAcc_Az)
        time.sleep(0.1)

        setSpeedAcc(1, (int((np.abs(old_Ze - new_Ze)/360*200*256)/travelTime)), maxAcc_Ze)
        time.sleep(0.1)
        
    # #Create String to send-----------------------------------------------------------------------------------------
    moveString_X = f"/3A{str(steps_X)}<CR>\r\n"
    moveString_Y = f"/4A{str(steps_Y)}<CR>\r\n"
    runString = f"/CR\r\n"


    #Command Motors--------------------------------------------------------------------------------------------------
    ser.write(moveString_X.encode('ascii'))
    time.sleep(0.1)
    ser.write(moveString_Y.encode('ascii'))
    time.sleep(0.1)
    ser.write(runString.encode('ascii'))
    time.sleep(0.1)
    moveTo_Ze(findTheta(x, y)*180/np.pi)
    time.sleep(0.1)
    if y != 275:
        moveTo_Az(int(-np.round(np.arctan2((x-275), (y-275))*6*3200/(2*np.pi)-(2/8)*6*3200)))
    else:
        moveTo_Az(int(-np.round((np.pi/2)*6*3200/(2*np.pi)-(2/8)*6*3200)))
    time.sleep(0.1)
    begin_Az()
    
def zero(motor):
    """
        Sets specified motor(s) be at zero at its current position
        Inputs:
          "_": all
          "C": Motors 3 and 4 (x-y stage)
            1: ze
            2: az
            3: X
            4: Y
            5: chop
    """
    if motor == 2:
        bus.send(can.Message(
        arbitration_id=CAN_ZP,
        data=struct.pack("<i", 0),
        is_extended_id=True
        ))
        return
        
    reset = f"/{motor}z0R\r\n"
    ser.write(reset.encode('ascii'))
    print(f"Motor {motor} has been zeroed")



def highZero(motor):
    """
        Sets specified motor(s) to be at a large number position for purposes of setting the zero value
        Inputs:
          "_": all
          "C": Motors 3 and 4 (x-y stage)
            1: ze
            2: az
            3: X
            4: Y
            5: chop
    """
    reset = f"/{motor}z100000000R\r\n"
    ser.write(reset.encode('ascii'))
    print(f"Motor {motor} has been set high")
    
def startChop(Hz):
    """
        turns on the chopper motor to turn indefinetly at a specified frequency
        Inputs: Hz: float: the frequency that the chopper will chop the hear source at (Hz)

        negetive input spins CW, positive spins CCW    
    """
    #set up chop
    initString_Chop = f"/5m{curr_Chop}V{str(np.abs(Hz*200*256//2))}L{maxAcc_Chop}R\r\n" #for velocity: Hz * full steps * microsteps * 1/2 (chopper wheel has 2 on/offs per rotation)
    ser.write(initString_Chop.encode("ascii"))
    time.sleep(0.5)
    if Hz > 0:
        go = f"/5P0<CR>\r\n"
    elif Hz < 0:
        go = f"/5D0<CR>\r\n"
    else:
        return
    runString = f"/5R\r\n"
    print(go)
    
    #Command Motors
    ser.write(go.encode('ascii'))
    time.sleep(0.5)
    ser.write(runString.encode('ascii'))
    time.sleep(0.5)
    print("Chop started")
    
def stopChop():
    """
        Stops chopper motor
    """
    runString = f"/5TR\r\n"
    ser.write(runString.encode('ascii'))

def setSpeedAcc(motor, speed, acc):
    """
        Sets the speed and acceleration for any given motor in the system
        Inputs:
            Motor: 
                "_": all
                "C": Motors 3 and 4 (x-y stage)
                1: ze
                2: az
                3: X
                4: Y
                5: chop
            Speed: float: value to set motor speed to in steps/sec
            Acc: float: value to set motor acceleration to in steps/sec^2
    """
    if motor == 2:
        setSpeed_Az(speed)
        time.sleep(0.1)
        setAcc_Az(acc)
        time.sleep(0.1)
        return
        
    initString = f"/{int(motor)}V{speed}L{acc}R\r\n"
    ser.write(initString.encode("ascii"))
    time.sleep(0.1)    

def readPosition(motor, timeout = 0.1):
    """
       gets the current position of any motor
       Inputs:
           motor: int: Motor for which the position is read
               1: ze
               2: az
               3: X
               4: Y
               5: chop
            timeout: float: time to wait in seconds before giving up on position read
        Outputs:
            Position: int: 
               1: ze in degrees
               2: az in degrees
               3: X in mm
               4: Y in mm
               5: chop in number of rotations
                
    """
    if motor == 2:
        bus.send(can.Message(
        arbitration_id=CAN_PA,
        data=[],
        is_extended_id=True
        ))
    
        t0 = time.time()
        while time.time() - t0 < timeout:
            msg = bus.recv(timeout=0.1)
            if msg is None:
                continue
    
            cw = msg.arbitration_id & 0xFF
    
            # ACK for position is CW = 0x1F
            if cw == 0x20 and msg.dlc >= 4:
                pos = struct.unpack("<i", msg.data[:4])[0]
                return int(pos/(6*16*200)*360)
    
        return None
        
    # Flush any leftover data in the buffers
    ser.reset_input_buffer()
    ser.reset_output_buffer()

    command = f"/{motor}?0R\r\n"
    ser.write(command.encode('ascii'))
    
    # Wait a brief moment for the controller to process and respond
    time.sleep(0.01)
    
    # Read the response line
    response = ser.readline().decode('ascii', errors='ignore').strip()
    
    if motor == 3 or motor == 4:     
        return int(response[3:-1])/(stepsPerRotation_XY/mmPerThread)
    elif motor == 5:
        return int(response[3:-1])/(200*256)
    elif motor ==1:
        return int(response[3:-1])/(200*256)*360
    

def endAll():
    """
        Stops all motors
    """
    stopMotion_Az()
    time.sleep(0.1)
    runString = f"/_TR\r\n"
    ser.write(runString.encode('ascii'))
    time.sleep(0.1)
    stopMotion_Az()
    time.sleep(0.1)

def close():
    """
        Closes serial port when done with beam mapper
    """
    ser.close()
    print(f"Port Closed")


# ----- CAN COMMAND SET -----
CAN_MO = 0x04000095   # Driver State (on/off)
CAN_SP = 0x0400009E   # PTP motor speed
CAN_PA = 0x040000A0   # Absolute position
CAN_BG = 0x04000096   # Begin Motion
CAN_ZP = 0x040000A1   # Set position (zero)
CAN_IC = 0x04000086   # Internal Configuration
CAN_PA = 0x040000A0   # Absolute Position
CAN_AC = 0x04000090   # Acceleration
CAN_DC = 0x04000091   # Deceleration
CAN_ST = 0x04000097   # Stop motion
CAN_JV = 0x0400009D   # Jog velocity
CAN_SY = 0x0400007E   # System Operation

def motorOn_Az():
    """
        turns the azimuth motor driver on
    """        
    bus.send(can.Message(
        arbitration_id=CAN_MO,
        data=[0x01],
        is_extended_id=True
    ))
    time.sleep(0.05)

def motorOff_Az():
    """
        turns the azimuth motor off
    """
    bus.send(can.Message(
        arbitration_id=CAN_MO,
        data=[0x00],
        is_extended_id=True
    ))

def enableClosedLoop_Az():
    """
        enables closed loop control of the azimuth motor
    """
    bus.send(can.Message(
        arbitration_id=CAN_IC,
        data=[0x06, 0x01, 0x00],
        is_extended_id=True
    ))
    time.sleep(0.1)

def setAcc_Az(accel):
    """
        sets the acceleration of the azimuth motor
        Inputs:
            accel: int: acceleration of motor (pulse/sec^2)
    """        
    bus.send(can.Message(
        arbitration_id=CAN_AC,
        data=struct.pack("<i", accel),
        is_extended_id=True
    ))

def setDecel_Az(decel):
    """
        sets the deceleration of the azimuth motor
        Inputs:
            decel: int: decceleration of motor (pulse/sec^2)
    """        
    bus.send(can.Message(
        arbitration_id=CAN_DC,
        data=struct.pack("<i", decel),
        is_extended_id=True
    ))
    time.sleep(0.05)

def zero_Az():
    """
        sets the current azimuth motor position to be the zero position
    """
    bus.send(can.Message(
        arbitration_id=CAN_ZP,
        data=struct.pack("<i", 0),
        is_extended_id=True
    ))

def moveTo_Az(position):
    """
        commanads the azimuth motor to move to a specific position
        Inputs:
            position: int: position to move to (steps) (3200 in one rotation)
    """
    bus.send(can.Message(
        arbitration_id=CAN_PA,
        data=struct.pack("<i", int(position)),
        is_extended_id=True
    ))
    print(f"Az command sent. Moving azimuth axis to {round(position/(6*16*200)*360, 3)} degrees")
    
def moveRelative_Az(position):
    """
        commanads the azimuth motor to move a certain amount
        Inputs:
            position: int: amount to move by (steps) (3200 in one rotation)
    """
    bus.send(can.Message(
        arbitration_id=CAN_PR,
        data=struct.pack("<i", int(position)),
        is_extended_id=True
    ))
    
def setSpeed_Az(speed):
    """
        sets the speed of the azimuth motor
        Inputs:
            speed: int: speed of motor (pulse/sec)
    """
    bus.send(can.Message(
        arbitration_id=CAN_SP,
        data=struct.pack("<i", int(speed)),
        is_extended_id=True
    ))
    
def begin_Az():
    """
        activates newly set parameters and initiates motion. must be ran for motor to move
    """
    bus.send(can.Message(
        arbitration_id=CAN_BG,
        data=[],
        is_extended_id=True
    ))


def stopMotion_Az():
    """
        stops current motor movement
    """
    bus.send(can.Message(
        arbitration_id=CAN_ST,
        data=[],
        is_extended_id=True
    ))

def clearJogMode_Az():
    """
        sets jog velocity to zero. Used to enter PTP mode if stuck
    """
    # JV = 0 → exit jog / velocity mode
    bus.send(can.Message(
        arbitration_id=CAN_JV,
        data=struct.pack("<i", 0),
        is_extended_id=True
    ))


def readPosition_Az(timeout=1.0):
    """
        Requests the current encoder position from the UIM342.
        Returns position (int) or None.
    """
    # INS: GET position (DL = 0)
    bus.send(can.Message(
        arbitration_id=CAN_PA,
        data=[],
        is_extended_id=True
    ))

    t0 = time.time()
    while time.time() - t0 < timeout:
        msg = bus.recv(timeout=0.1)
        if msg is None:
            continue

        cw = msg.arbitration_id & 0xFF

        # ACK for position is CW = 0x1F
        if cw == 0x20 and msg.dlc >= 4:
            pos = struct.unpack("<i", msg.data[:4])[0]
            return pos

    return None

def invertEncoderDirection_Az():
    """
        Inverts the encoder direction of the azimuth motor. USE WITH CAUTION!!!
    """
    # IC[5] = 1 → invert encoder polarity
    bus.send(can.Message(
        arbitration_id=CAN_IC,
        data=[0x05, 0x01, 0x00],
        is_extended_id=True
    ))

def restoreFactoryDefaults_Az():
    bus.send(can.Message(
        arbitration_id=CAN_SY,
        data=[0x02],
        is_extended_id=True
    ))
# -----------------------------    
    

In [532]:
port_CAN = 'COM5' # Change according to your computer. Look in Device Manager
port_RS485 = 'COM3' # Change according to your computer. Look in Device Manager

baud = 9600 # for RS485
BITRATE = 500000 #for CAN

#Initialization for RS485 Port
ser = serial.Serial(port_RS485, baudrate = baud, timeout = 1)
time.sleep(0.5)
ser.reset_input_buffer()
ser.reset_output_buffer()

#initiates CAN bus port
bus = can.interface.Bus(
    interface="slcan",
    channel=port_CAN,
    bitrate=BITRATE
    )
motorOn_Az()
print("Ports active")

Ports active


In [703]:

#------------------------------------------------------------------------------------
#------------------------------------------------------------------------------------

#Motor Numbers:
#   Zenith: 1 6*200*16 for one fuill rotation of az-ze stage
#   Azimuth: N/A
#   Chop: 2
#   X: 3
#   Y: 4

stepsPerRotation_XY = 200*256 #full steps * number of microsteps
mmPerThread = 5 # 5 was roughly measured (dont ask Owen about this)


maxVel_X = 90000 #Microsteps/Second (90000 MAX works for 256 microsteps) (int)
maxAcc_X = 50 #Microsteps/Second^2 / (400,000,000/65536) (int)
curr_X = 100 #Movement Current as a percentage of max (Typically Use 100) (int)

maxVel_Y = 200000 #Microsteps/Second (400000 MAX works for 256 microsteps) (int)
maxAcc_Y = 50 #Microsteps/Second^2 / (400,000,000/65536) (int)
curr_Y = 100 #Movement Current as a percentage of max (Typically Use 100) (int)

maxVel_Ze = 50000 #Microsteps/Second (50000 works for 256 microsteps) (int)
maxAcc_Ze = 100 #Microsteps/Second^2 / (400,000,000/65536)  (int)
curr_Ze = 100 #Movement Current as a percentage of max (Typically Use 100) (int)

maxVel_Az = 32000 #Microsteps/Second (32000 works fast and safe for 16 microsteps) (int)
maxAcc_Az = 100 #Microsteps/Second^2  (int) FIGURE THIS OUT
# curr_Az = 2 #current working current (not sure how this effects movement so leaving it at the deffault for now) (int)

curr_Chop = 100 #Movement Current as a percentage of max (Typically Use 100) (int)
maxAcc_Chop = 100 #Microsteps/Second^2 / (400,000,000/65536) (int)

k = 10 #heat source height above zenith rotation axis (mm)
g = 5 #distance of heat source from center of azimuth rotation axis (mm)
h = 300 #vertical distnace from center of zenith rotation axis to point where the heat source should be aimed at (mm)

#------------------------------------------------------------------------------------
#------------------------------------------------------------------------------------

initString_X = f"/3m{curr_X}V{maxVel_X}L{maxAcc_X}R\r\n"
initString_Y = f"/4m{curr_Y}V{maxVel_Y}L{maxAcc_Y}R\r\n"
initString_Ze = f"/1m{curr_Ze}V{maxVel_Ze}L{maxAcc_Ze}R\r\n"

#Initialization for RS485 Port
ser.write(initString_X.encode("ascii"))
time.sleep(0.1)
ser.write(initString_Y.encode("ascii"))
time.sleep(0.1)
ser.write(initString_Ze.encode("ascii"))
time.sleep(0.1)
print("Initiation Complete")


setAcc_Az(maxAcc_Az)
setSpeed_Az(maxVel_Az)

Initiation Complete


In [534]:
#math Functions

def findTheta(X_A, Y_A, K = k, G = g, H = h):
    """
        determines the amount to angle the heat source given its postion
        Inputs:
            X_A: float: X position (mm)
            X_A: float: Y position (mm)
        Outputs:
            theta: float: Angle of heat source in radians
    """
        
    def thetaFunc(theta, X_A, Y_A, K, H):
        return np.arctan((((((X_A-275)**2)+((Y_A-275)**2))**0.5)-K*np.sin(theta))/(H-K*np.cos(theta)))-theta
        
    return opt.brentq(thetaFunc, -np.pi, np.pi/2, args = (X_A, Y_A, K, H))



In [617]:
print(findTheta(0, 0)*180/np.pi)

52.35379303249375


In [530]:
close()
bus.shutdown()

Port Closed


In [728]:
highZero(4)

Motor 4 has been set high


In [731]:
zero("C")

Motor C has been zeroed


In [718]:
moveTo_XY(50, 100)

XY command sent. Moving to (50, 100)


In [699]:
print(readPosition(1))

90.0


In [730]:
moveRelative_XY(0, -10)

XY command sent. Moving by (0, -10) to (0.0001953125, 9725.625)


In [700]:
moveRelative_Ze(45)

Ze command sent. Moving zenith axis by 45 degrees


In [701]:
moveTo_Ze(45)

Ze command sent. Moving zenith axis to 45 degrees


In [492]:
# string = f"/1?9R\r\n"
# ser.write(string.encode('ascii'))

7

In [281]:
moveTo_all(100, 0)

Ze command sent. Moving zenith axis to 42.675 degrees
Az command sent. Moving azimuth axis to 237.525 degrees


In [723]:
endAll()

In [675]:
startChop(-90)

/5D0<CR>

Chop started


In [680]:
stopChop()

In [433]:
zero(1)

Motor 1 has been zeroed


In [685]:
motorOn_Az()
setSpeedAcc(2, 2*maxVel_Az, maxAcc_Az)
moveTo_Az(1*6*200*16)
begin_Az()

Az command sent. Moving azimuth axis to 360.0 degrees


In [677]:
motorOn_Az()

In [32]:
zero_Az()

In [683]:
readPosition_Az()

In [269]:
print(readPosition(5))

271.751015625


In [719]:
def moveTo_all(x, y):
    """
        Moves all motors so that the the heat source is pointed towards a target some distance (H defined below) above (x, y) = (275, 275)
        Inputs:
            x: float: X location to move to in mm (0-550)
            y: float: Y location to move to in mm (0-550)
    """
    #Calculating steps based on distance and physical properties of rails---------------------------------------------
    steps_X = int(np.abs(x*stepsPerRotation_XY/mmPerThread))
    steps_Y = int(np.abs(y*stepsPerRotation_XY/mmPerThread))

    # matching Move times---------------------------------------------------------------------------------------------
    if (np.abs(x-readPosition(3))*(np.abs(y-readPosition(4))) != 0):
        speed_X = int(np.min(np.array([maxVel_X, maxVel_Y*(np.abs(x-readPosition(3))/np.abs(y-readPosition(4)))])))
        speed_Y = int(np.min(np.array([maxVel_Y, maxVel_X*(np.abs(y-readPosition(4))/np.abs(x-readPosition(3)))])))
        acc_X = int(np.min(np.array([maxAcc_X, maxAcc_Y*speed_X/speed_Y])))
        acc_Y = int(np.min(np.array([maxAcc_Y, maxAcc_X*speed_Y/speed_X])))
        setSpeedAcc(3, speed_X, acc_X)
        setSpeedAcc(4, speed_Y, acc_Y)
    elif (np.abs(x-readPosition(3)) == 0):
        setSpeedAcc(4, maxVel_Y, maxAcc_Y)
    elif (np.abs(y-readPosition(4)) == 0):
        setSpeedAcc(3, maxVel_X, maxAcc_X)
        
    # #Create String to send-----------------------------------------------------------------------------------------
    moveString_X = f"/3A{str(steps_X)}<CR>\r\n"
    moveString_Y = f"/4A{str(steps_Y)}<CR>\r\n"
    runString = f"/CR\r\n"


    #Command Motors--------------------------------------------------------------------------------------------------
    ser.write(moveString_X.encode('ascii'))
    time.sleep(0.1)
    ser.write(moveString_Y.encode('ascii'))
    time.sleep(0.1)
    ser.write(runString.encode('ascii'))
    time.sleep(0.1)

    setSpeedAcc(2, int(maxVel_Az), int(maxAcc_Az))
    setSpeedAcc(1, int(maxVel_Ze), int(maxAcc_Ze))

    while (x != readPosition(3)) or (y != readPosition(4)):
        moveTo_Ze(findTheta(readPosition(3), readPosition(4))*180/np.pi)
        
        if readPosition(4) != 275:
            moveTo_Az(int(-np.round(np.arctan2((readPosition(3)-275), (readPosition(4)-275))*6*3200/(2*np.pi)-(2/8)*6*3200)))
        else:
            moveTo_Az(int(-np.round((np.pi/2)*6*3200/(2*np.pi)-(2/8)*6*3200)))
        begin_Az()

    moveTo_Ze(findTheta(x, y)*180/np.pi)
    if readPosition(4) != 275:
        moveTo_Az(int(-np.round(np.arctan2((x-275), (y-275))*6*3200/(2*np.pi)-(2/8)*6*3200)))
    else:
        moveTo_Az(int(-np.round((np.pi/2)*6*3200/(2*np.pi)-(2/8)*6*3200)))
    begin_Az()
    
    
moveTo_all(0, 0)

Ze command sent. Moving zenith axis to 43.845 degrees
Az command sent. Moving azimuth axis to 218.531 degrees
Ze command sent. Moving zenith axis to 44.339 degrees
Az command sent. Moving azimuth axis to 218.981 degrees
Ze command sent. Moving zenith axis to 44.834 degrees
Az command sent. Moving azimuth axis to 219.431 degrees
Ze command sent. Moving zenith axis to 45.318 degrees
Az command sent. Moving azimuth axis to 219.844 degrees
Ze command sent. Moving zenith axis to 45.799 degrees
Az command sent. Moving azimuth axis to 220.256 degrees
Ze command sent. Moving zenith axis to 46.268 degrees
Az command sent. Moving azimuth axis to 220.65 degrees
Ze command sent. Moving zenith axis to 46.731 degrees
Az command sent. Moving azimuth axis to 221.025 degrees
Ze command sent. Moving zenith axis to 47.187 degrees
Az command sent. Moving azimuth axis to 221.4 degrees
Ze command sent. Moving zenith axis to 47.64 degrees
Az command sent. Moving azimuth axis to 221.775 degrees
Ze command sen